# STQD6324 Data Management — Assignment 1
## P166248 Suraya Balqis binti Ab.Ghafar
### Iris Dataset Classification using Spark MLlib

---

### Overview
**Apache Spark MLlib** is used to build and evaluate three classification models on the **Iris dataset**.

### **SECTION 1: Load Iris Dataset into Spark DataFrame**
#### The Iris dataset is loaded using sklearn and converted to a Spark DataFrame for MLlib processing.
#### This notebook is generated from Databricks. In Databricks, the SparkSession 'spark' is already available.
#### In standalone PySpark, we would use:
##### from pyspark.sql import SparkSession
##### spark = SparkSession.builder.appName("IrisExample").getOrCreate()

In [0]:
# SECTION 1A: Load Iris Dataset into Spark DataFrame

# Import Libraries
from sklearn.datasets import load_iris
import pandas as pd

# Load Iris dataset
iris = load_iris(as_frame=True)
iris

# Combine features and labels
iris_df = pd.concat([iris.data, iris.target.rename("label")], axis=1)

# Convert to Spark DataFrame
df = spark.createDataFrame(iris_df)

# Display the DataFrame
display(df)


### **SECTION 1B: Exploratory Data Analysis (EDA)**

Exploratory Data Analysis helps us understand the Iris dataset structure, distributions, relationships, and potential issues before building models.

#### **What we'll explore:**
* Dataset shape and structure
* Statistical summary
* Class distribution
* Feature distributions
* Correlation analysis
* Outlier detection

In [0]:
# SECTION 1B: Exploratory Data Analysis (EDA)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("="*70)
print("EXPLORATORY DATA ANALYSIS - IRIS DATASET")
print("="*70)

# Convert to Pandas for easier EDA
iris_pd = df.toPandas()

print("\n[1] DATASET OVERVIEW")
print("="*70)
print(f"Dataset Shape: {iris_pd.shape[0]} rows × {iris_pd.shape[1]} columns")
print(f"\nColumn Names and Types:")
print(iris_pd.dtypes)

print("\n[2] STATISTICAL SUMMARY")
print("="*70)
print(iris_pd.describe())

print("\n[3] MISSING VALUES CHECK")
print("="*70)
print(iris_pd.isnull().sum())
if iris_pd.isnull().sum().sum() == 0:
    print("✓ No missing values found!")

In [0]:
# Class Distribution
print("\n" + "="*70)
print("[4] CLASS DISTRIBUTION")
print("="*70)

class_counts = iris_pd['label'].value_counts().sort_index()
print("\nClass counts:")
for label, count in class_counts.items():
    print(f"  Class {label}: {count} samples ({count/len(iris_pd)*100:.1f}%)")

# Visualize class distribution
fig, ax = plt.subplots(figsize=(8, 5))
iris_pd['label'].value_counts().sort_index().plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_title('Class Distribution in Iris Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Class Label (0=Setosa, 1=Versicolor, 2=Virginica)', fontsize=11)
ax.set_ylabel('Number of Samples', fontsize=11)
ax.set_xticklabels(['Setosa (0)', 'Versicolor (1)', 'Virginica (2)'], rotation=0)
plt.tight_layout()
plt.show()

print("\n✓ Dataset is perfectly balanced: 50 samples per class")

In [0]:
# Feature Distributions
print("\n" + "="*70)
print("[5] FEATURE DISTRIBUTIONS")
print("="*70)

# Create histograms for all features
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
feature_cols = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]
colors = ['skyblue', 'lightcoral', 'lightgreen', 'plum']

for idx, (col, color) in enumerate(zip(feature_cols, colors)):
    ax = axes[idx // 2, idx % 2]
    iris_pd[col].hist(bins=20, ax=ax, color=color, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col.title()} Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.axvline(iris_pd[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {iris_pd[col].mean():.2f}')
    ax.legend()

plt.tight_layout()
plt.show()

print("\n✓ Feature distributions show clear patterns for classification")

In [0]:
# Correlation Analysis
print("\n" + "="*70)
print("[6] CORRELATION ANALYSIS")
print("="*70)

# Calculate correlation matrix
corr_matrix = iris_pd.corr()
print("\nCorrelation Matrix:")
print(corr_matrix)

# Visualize correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n✓ Key observations:")
print("  - Petal length & petal width are highly correlated (0.96)")
print("  - Sepal measurements show moderate correlation with petal measurements")
print("  - Sepal width has weaker correlations with other features")

In [0]:
# Pairwise Relationships
print("\n" + "="*70)
print("[7] PAIRWISE FEATURE RELATIONSHIPS")
print("="*70)

# Create pairplot
class_names = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}
iris_pd['class_name'] = iris_pd['label'].map(class_names)

fig = sns.pairplot(iris_pd, hue='class_name', 
                   vars=["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"],
                   diag_kind='hist', 
                   palette={'Setosa': '#1f77b4', 'Versicolor': '#ff7f0e', 'Virginica': '#2ca02c'},
                   plot_kws={'alpha': 0.6, 's': 50, 'edgecolor': 'k'})
fig.fig.suptitle('Pairwise Feature Relationships by Class', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ Pairplot reveals:")
print("  - Setosa is clearly separable from other classes")
print("  - Versicolor and Virginica have some overlap")
print("  - Petal measurements provide strongest class separation")

In [0]:
# Box Plots for Outlier Detection
print("\n" + "="*70)
print("[8] OUTLIER DETECTION (BOX PLOTS)")
print("="*70)

# Create box plots for all features
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
feature_cols = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]
colors = ['skyblue', 'lightcoral', 'lightgreen', 'plum']

for idx, (col, color) in enumerate(zip(feature_cols, colors)):
    ax = axes[idx // 2, idx % 2]
    iris_pd.boxplot(column=col, by='class_name', ax=ax, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'{col.title()} by Class', fontsize=12, fontweight='bold')
    ax.set_xlabel('Class', fontsize=10)
    ax.set_ylabel(col, fontsize=10)
    plt.sca(ax)
    plt.xticks(rotation=0)

plt.suptitle('')  # Remove default title
fig.suptitle('Feature Distributions by Class (Outlier Detection)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n✓ Minimal outliers detected - Iris dataset is clean and well-structured")

#### EDA SUMMARY - KEY INSIGHTS

✓ Dataset Characteristics:
  • 150 samples, 4 features, 3 balanced classes (50 each)
  • No missing values or data quality issues
  • Features are numeric and well-scaled

✓ Feature Insights:
  • Petal length & width are highly correlated (0.96)
  • Petal measurements show strongest class separation
  • Sepal width has weakest discriminative power

✓ Class Separation:
  • Setosa is linearly separable from other classes
  • Versicolor and Virginica have moderate overlap
  • Multi-class classification is feasible

✓ Data Quality:
  • Minimal outliers detected
  • Distributions are relatively normal
  • Dataset is ideal for classification tasks

### SECTION 2: Data Preprocessing
#### Features are assembled and the label column is indexed for MLlib compatibility.

#### This shows how to prepare raw Iris data so Spark MLlib can understand it by turning multiple numeric columns into a single feature vector.


In [0]:
# SECTION 2: Data Preprocessing
# Features are assembled into a single vector column for MLlib compatibility
# Note: StringIndexer is NOT needed since species labels are already numerical (0, 1, 2)

from pyspark.ml.feature import VectorAssembler

# Define feature columns
feature_cols = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]

# Assemble features into a single vector column
# This combines all four numeric measurements into a single "features" vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Apply the transformation
df_assembled = assembler.transform(df)

display(df_assembled.select("features", "label"))

### **SECTION 3: Split Dataset**
#### The dataset is split into training (70%) and testing (30%) sets for model evaluation.
#### Training set (70%) is used to train the models, letting them learn patterns from the data
#### Testing set (30%) is kept separate to evaluate model performance on unseen data - give unbiased assessment how well the model generalizes

In [0]:
# SECTION 3: Split Dataset
# Split into training (70%) and testing (30%) sets with a fixed seed for reproducibility

train_df, test_df = df_assembled.randomSplit([0.7, 0.3], seed=42)

print(f"Training set size: {train_df.count()}")
print(f"Testing set size: {test_df.count()}")

### **SECTION 4: Random Forest Classifier with Hyperparameter Tuning**

#### **What is Random Forest?**
An ensemble method that creates multiple decision trees and combines their predictions through voting.

#### **Hyperparameter Tuning Approach:**
* **Manual Grid Search**: Tests all combinations of hyperparameters systematically
* **Validation Split**: Uses 80% of training data for model fitting, 20% for validation
* **Parameters tuned**: `numTrees` (number of trees) and `maxDepth` (tree depth limit)
* **Selection Criterion**: F1 score on validation set

In [0]:
# SECTION 4: Random Forest Classifier with Hyperparameter Tuning

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("="*60)
print("RANDOM FOREST CLASSIFIER")
print("="*60)

# Step 1: Split training data for validation
train_data, val_data = train_df.randomSplit([0.8, 0.2], seed=42)

# Step 2: Define hyperparameter grid
num_trees_options = [10, 20, 30]
max_depth_options = [3, 5, 7]

print(f"\n[1] Testing hyperparameter combinations:")
print(f"    - numTrees: {num_trees_options}")
print(f"    - maxDepth: {max_depth_options}")
print(f"    - Total combinations: {len(num_trees_options) * len(max_depth_options)}")

# Step 3: Grid search - test all combinations
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

print("\n[2] Running grid search on validation set...")
best_score = 0
best_params = {}

for num_trees in num_trees_options:
    for max_depth in max_depth_options:
        # Train model with current parameters
        rf_temp = RandomForestClassifier(
            labelCol="label",
            featuresCol="features",
            numTrees=num_trees,
            maxDepth=max_depth,
            seed=42
        )
        rf_temp_model = rf_temp.fit(train_data)
        
        # Evaluate on validation set
        predictions = rf_temp_model.transform(val_data)
        score = evaluator.evaluate(predictions)
        
        # Update best parameters if this is better
        if score > best_score:
            best_score = score
            best_params = {'numTrees': num_trees, 'maxDepth': max_depth}

print(f"\n[3] Best hyperparameters found (validation F1: {best_score:.4f}):")
print(f"    - numTrees: {best_params['numTrees']}")
print(f"    - maxDepth: {best_params['maxDepth']}")

# Step 4: Retrain with best parameters on full training set
print("\n[4] Retraining with best parameters on full training set...")
rf_best_model = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=best_params['numTrees'],
    maxDepth=best_params['maxDepth'],
    seed=42
).fit(train_df)

# Step 5: Evaluate on test set
rf_predictions = rf_best_model.transform(test_df)

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

rf_accuracy = evaluator_acc.evaluate(rf_predictions)
rf_precision = evaluator_prec.evaluate(rf_predictions)
rf_recall = evaluator_rec.evaluate(rf_predictions)
rf_f1 = evaluator_f1.evaluate(rf_predictions)

print("\n[5] Performance on test set:")
print(f"    - Accuracy:  {rf_accuracy:.4f}")
print(f"    - Precision: {rf_precision:.4f}")
print(f"    - Recall:    {rf_recall:.4f}")
print(f"    - F1 Score:  {rf_f1:.4f}")
print("="*60)

### **SECTION 5: Logistic Regression Classifier with Hyperparameter Tuning**

#### **What is Logistic Regression?**
A linear model that estimates probabilities for each class using mathematical functions.

#### **Hyperparameter Tuning Approach:**
* **Manual Grid Search**: Tests all combinations of hyperparameters systematically
* **Validation Split**: Uses 80% of training data for model fitting, 20% for validation
* **Parameters tuned**: `regParam` (regularization strength) and `maxIter` (maximum iterations)
* **Selection Criterion**: F1 score on validation set

In [0]:
# SECTION 5: Logistic Regression Classifier with Hyperparameter Tuning

from pyspark.ml.classification import LogisticRegression

print("="*60)
print("LOGISTIC REGRESSION CLASSIFIER")
print("="*60)

# Step 1: Define hyperparameter grid
reg_param_options = [0.001, 0.01, 0.1]
max_iter_options = [50, 100, 150]

print(f"\n[1] Testing hyperparameter combinations:")
print(f"    - regParam: {reg_param_options}")
print(f"    - maxIter: {max_iter_options}")
print(f"    - Total combinations: {len(reg_param_options) * len(max_iter_options)}")

# Step 2: Grid search - test all combinations
print("\n[2] Running grid search on validation set...")
best_score = 0
best_params = {}

for reg_param in reg_param_options:
    for max_iter in max_iter_options:
        # Train model with current parameters
        lr_temp = LogisticRegression(
            labelCol="label",
            featuresCol="features",
            regParam=reg_param,
            maxIter=max_iter
        )
        lr_temp_model = lr_temp.fit(train_data)
        
        # Evaluate on validation set
        predictions = lr_temp_model.transform(val_data)
        score = evaluator.evaluate(predictions)
        
        # Update best parameters if this is better
        if score > best_score:
            best_score = score
            best_params = {'regParam': reg_param, 'maxIter': max_iter}

print(f"\n[3] Best hyperparameters found (validation F1: {best_score:.4f}):")
print(f"    - regParam: {best_params['regParam']}")
print(f"    - maxIter: {best_params['maxIter']}")

# Step 3: Retrain with best parameters on full training set
print("\n[4] Retraining with best parameters on full training set...")
lr_best_model = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    regParam=best_params['regParam'],
    maxIter=best_params['maxIter']
).fit(train_df)

# Step 4: Evaluate on test set
lr_predictions = lr_best_model.transform(test_df)

lr_accuracy = evaluator_acc.evaluate(lr_predictions)
lr_precision = evaluator_prec.evaluate(lr_predictions)
lr_recall = evaluator_rec.evaluate(lr_predictions)
lr_f1 = evaluator_f1.evaluate(lr_predictions)

print("\n[5] Performance on test set:")
print(f"    - Accuracy:  {lr_accuracy:.4f}")
print(f"    - Precision: {lr_precision:.4f}")
print(f"    - Recall:    {lr_recall:.4f}")
print(f"    - F1 Score:  {lr_f1:.4f}")
print("="*60)

### **SECTION 6: Decision Tree Classifier with Hyperparameter Tuning**

#### **What is Decision Tree?**
A single tree that makes decisions by splitting data based on feature values in a flowchart-like structure.

#### **Hyperparameter Tuning Approach:**
* **Manual Grid Search**: Tests all combinations of hyperparameters systematically
* **Validation Split**: Uses 80% of training data for model fitting, 20% for validation
* **Parameters tuned**: `maxDepth` (tree depth limit) and `minInstancesPerNode` (minimum samples per node)
* **Selection Criterion**: F1 score on validation set

In [0]:
# SECTION 6: Decision Tree Classifier with Hyperparameter Tuning

from pyspark.ml.classification import DecisionTreeClassifier

print("="*60)
print("DECISION TREE CLASSIFIER")
print("="*60)

# Step 1: Define hyperparameter grid
max_depth_options = [3, 5, 7, 10]
min_instances_options = [1, 5, 10]

print(f"\n[1] Testing hyperparameter combinations:")
print(f"    - maxDepth: {max_depth_options}")
print(f"    - minInstancesPerNode: {min_instances_options}")
print(f"    - Total combinations: {len(max_depth_options) * len(min_instances_options)}")

# Step 2: Grid search - test all combinations
print("\n[2] Running grid search on validation set...")
best_score = 0
best_params = {}

for max_depth in max_depth_options:
    for min_instances in min_instances_options:
        # Train model with current parameters
        dt_temp = DecisionTreeClassifier(
            labelCol="label",
            featuresCol="features",
            maxDepth=max_depth,
            minInstancesPerNode=min_instances,
            seed=42
        )
        dt_temp_model = dt_temp.fit(train_data)
        
        # Evaluate on validation set
        predictions = dt_temp_model.transform(val_data)
        score = evaluator.evaluate(predictions)
        
        # Update best parameters if this is better
        if score > best_score:
            best_score = score
            best_params = {'maxDepth': max_depth, 'minInstancesPerNode': min_instances}

print(f"\n[3] Best hyperparameters found (validation F1: {best_score:.4f}):")
print(f"    - maxDepth: {best_params['maxDepth']}")
print(f"    - minInstancesPerNode: {best_params['minInstancesPerNode']}")

# Step 3: Retrain with best parameters on full training set
print("\n[4] Retraining with best parameters on full training set...")
dt_best_model = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=best_params['maxDepth'],
    minInstancesPerNode=best_params['minInstancesPerNode'],
    seed=42
).fit(train_df)

# Step 4: Evaluate on test set
dt_predictions = dt_best_model.transform(test_df)

dt_accuracy = evaluator_acc.evaluate(dt_predictions)
dt_precision = evaluator_prec.evaluate(dt_predictions)
dt_recall = evaluator_rec.evaluate(dt_predictions)
dt_f1 = evaluator_f1.evaluate(dt_predictions)

print("\n[5] Performance on test set:")
print(f"    - Accuracy:  {dt_accuracy:.4f}")
print(f"    - Precision: {dt_precision:.4f}")
print(f"    - Recall:    {dt_recall:.4f}")
print(f"    - F1 Score:  {dt_f1:.4f}")
print("="*60)

### **SECTION 6B: Multilayer Perceptron Classifier with Hyperparameter Tuning**

#### **What is Multilayer Perceptron?**
A feedforward artificial neural network with multiple layers that can learn complex non-linear patterns through backpropagation.

#### **Hyperparameter Tuning Approach:**
* **Manual Grid Search**: Tests different network architectures
* **Validation Split**: Uses 80% of training data for model fitting, 20% for validation
* **Parameter tuned**: `layers` (network architecture: input, hidden layers, output)
* **Selection Criterion**: F1 score on validation set
* **Note**: Input layer size = 4 (features), Output layer size = 3 (classes)

In [0]:
# SECTION 6B: Multilayer Perceptron Classifier with Hyperparameter Tuning

from pyspark.ml.classification import MultilayerPerceptronClassifier

print("="*60)
print("MULTILAYER PERCEPTRON CLASSIFIER")
print("="*60)

# Step 1: Define hyperparameter grid (network architectures)
# Format: [input_layer, hidden_layer1, hidden_layer2, ..., output_layer]
# Input = 4 features, Output = 3 classes
layers_options = [
    [4, 5, 3],      # 1 hidden layer with 5 neurons
    [4, 10, 3],     # 1 hidden layer with 10 neurons
    [4, 5, 5, 3],   # 2 hidden layers with 5 neurons each
    [4, 10, 5, 3]   # 2 hidden layers with 10 and 5 neurons
]

print(f"\n[1] Testing hyperparameter combinations:")
print(f"    - Network architectures (layers): {len(layers_options)} options")
for i, layers in enumerate(layers_options, 1):
    print(f"      {i}. {layers}")

# Step 2: Grid search - test all combinations
print("\n[2] Running grid search on validation set...")
best_score = 0
best_params = {}

for layers in layers_options:
    # Train model with current parameters
    mlp_temp = MultilayerPerceptronClassifier(
        labelCol="label",
        featuresCol="features",
        layers=layers,
        maxIter=100,
        seed=42
    )
    mlp_temp_model = mlp_temp.fit(train_data)
    
    # Evaluate on validation set
    predictions = mlp_temp_model.transform(val_data)
    score = evaluator.evaluate(predictions)
    
    # Update best parameters if this is better
    if score > best_score:
        best_score = score
        best_params = {'layers': layers}

print(f"\n[3] Best hyperparameters found (validation F1: {best_score:.4f}):")
print(f"    - layers: {best_params['layers']}")

# Step 3: Retrain with best parameters on full training set
print("\n[4] Retraining with best parameters on full training set...")
mlp_best_model = MultilayerPerceptronClassifier(
    labelCol="label",
    featuresCol="features",
    layers=best_params['layers'],
    maxIter=100,
    seed=42
).fit(train_df)

# Step 4: Evaluate on test set
mlp_predictions = mlp_best_model.transform(test_df)

mlp_accuracy = evaluator_acc.evaluate(mlp_predictions)
mlp_precision = evaluator_prec.evaluate(mlp_predictions)
mlp_recall = evaluator_rec.evaluate(mlp_predictions)
mlp_f1 = evaluator_f1.evaluate(mlp_predictions)

print("\n[5] Performance on test set:")
print(f"    - Accuracy:  {mlp_accuracy:.4f}")
print(f"    - Precision: {mlp_precision:.4f}")
print(f"    - Recall:    {mlp_recall:.4f}")
print(f"    - F1 Score:  {mlp_f1:.4f}")
print("="*60)

### **SECTION 7: Compare All Four Classification Models**
#### Comprehensive comparison of Random Forest, Logistic Regression, Decision Tree, and Multilayer Perceptron showing which performs best on the Iris dataset.

In [0]:
# SECTION 7: Compare All Four Classification Models
import pandas as pd

print("\n" + "="*70)
print("FINAL MODEL COMPARISON")
print("Random Forest vs Logistic Regression vs Decision Tree vs MLP")
print("="*70)

# Create comparison dataframe
results = pd.DataFrame([
    {
        'Model': 'Random Forest',
        'Accuracy': rf_accuracy,
        'Precision': rf_precision,
        'Recall': rf_recall,
        'F1 Score': rf_f1
    },
    {
        'Model': 'Logistic Regression',
        'Accuracy': lr_accuracy,
        'Precision': lr_precision,
        'Recall': lr_recall,
        'F1 Score': lr_f1
    },
    {
        'Model': 'Decision Tree',
        'Accuracy': dt_accuracy,
        'Precision': dt_precision,
        'Recall': dt_recall,
        'F1 Score': dt_f1
    },
    {
        'Model': 'Multilayer Perceptron',
        'Accuracy': mlp_accuracy,
        'Precision': mlp_precision,
        'Recall': mlp_recall,
        'F1 Score': mlp_f1
    }
])

# Sort by F1 Score descending
results = results.sort_values('F1 Score', ascending=False).reset_index(drop=True)

print("\n" + results.to_string(index=False))

# Identify best model
best_model_name = results.iloc[0]['Model']
best_f1_score = results.iloc[0]['F1 Score']

print("\n" + "="*70)
print(f"🏆 BEST MODEL: {best_model_name} (F1 Score: {best_f1_score:.4f})")
print("="*70)

# Show performance summary
print("\n📊 PERFORMANCE SUMMARY:")
print(f"   Highest Accuracy:  {results.iloc[0]['Model']} ({results.iloc[0]['Accuracy']:.4f})")
print(f"   Average F1 Score:  {results['F1 Score'].mean():.4f}")
print(f"   Performance Range: {results['F1 Score'].min():.4f} - {results['F1 Score'].max():.4f}")

### **SECTION 8: Display Predictions from Best Model**
#### Sample predictions showing how the best model classifies iris flowers.

In [0]:
# SECTION 8: Display Predictions from Best Model

print("\nSample predictions from the best performing model:")
print("(Showing features, actual label, and predicted label)\n")

# Display predictions from Random Forest (typically the best)
display(rf_predictions.select("features", "label", "prediction", "probability").limit(20))